In [1]:
# ===== CELL 1: SETUP =====
import pandas as pd
import numpy as np
import os
import re
import sqlite3
import joblib

RAW_DIR = "data/raw"
DB_PATH = "data/startup_intelligence.db"
conn = sqlite3.connect(DB_PATH)

log_reg = joblib.load("models/investment_readiness_model.pkl")
scaler = joblib.load("models/feature_scaler.pkl")
feature_columns = joblib.load("models/feature_columns.pkl")

print("Setup complete.")

Setup complete.


In [7]:
!pip install reportlab --quiet

In [2]:
# Is df_openvc loaded?
print('df_openvc' in dir())
print('compute_investor_fit' in dir())

False
False


In [3]:
# ===== CELL 2: LOAD OPENVC + MATCHING FUNCTIONS (from notebook 04) =====

df_openvc = pd.read_csv(os.path.join(RAW_DIR, "openvc_investors.csv"))

def parse_check_size(val):
    if pd.isna(val):
        return None
    val = str(val).strip().upper()
    match = re.search(r'([\d.]+)\s*([KM])', val)
    if not match:
        return None
    num, unit = match.groups()
    num = float(num)
    return num * 1000 if unit == 'K' else num * 1_000_000

def parse_stages(val):
    if pd.isna(val):
        return []
    stages = [s.strip() for s in str(val).split(',')]
    stages = [re.sub(r'^\d+\.\s*', '', s) for s in stages]
    return stages

df_openvc['check_size_usd'] = df_openvc['check_size'].apply(parse_check_size)
df_openvc['stages_list'] = df_openvc['stages'].apply(parse_stages)

SECTOR_KEYWORDS = {
    'Gaming & Entertainment': ['gaming', 'game', 'entertainment', 'media', 'interactive'],
    'Artificial Intelligence': ['ai', 'artificial intelligence', 'machine learning', 'ml'],
    'Fintech': ['fintech', 'financial', 'banking', 'payments'],
    'Healthcare & Biotech': ['health', 'healthcare', 'biotech', 'medical', 'life sciences'],
    'SaaS / Enterprise Software': ['saas', 'enterprise software', 'b2b software', 'enterprise'],
    'E-Commerce': ['e-commerce', 'ecommerce', 'commerce', 'retail', 'consumer'],
    'Cybersecurity': ['cybersecurity', 'security', 'infosec'],
    'Climate Tech / Clean Energy': ['climate', 'clean energy', 'sustainability', 'cleantech'],
    'Food & AgriTech': ['food', 'agritech', 'agriculture', 'agtech'],
    'Robotics & Automation': ['robotics', 'automation', 'hardware'],
    'Space Tech': ['space', 'aerospace'],
    'HR Tech': ['hr tech', 'human resources', 'talent', 'future of work'],
    'Logistics & Supply Chain': ['logistics', 'supply chain', 'shipping'],
    'Legal Tech': ['legal', 'legaltech', 'compliance'],
    'Real Estate Tech': ['real estate', 'proptech'],
    'Social Media / Creator Economy': ['social', 'creator economy', 'community', 'consumer'],
    'Insurtech': ['insurtech', 'insurance'],
    'Travel Tech': ['travel', 'hospitality'],
    'Edtech': ['edtech', 'education', 'learning'],
    'Web3 / Blockchain': ['web3', 'blockchain', 'crypto', 'defi']
}

def compute_investor_fit(startup_sector, startup_stage, startup_funding_needed_usd, investors_df):
    results = []
    keywords = SECTOR_KEYWORDS.get(startup_sector, [startup_sector.lower()])
    for _, inv in investors_df.iterrows():
        score = 0
        reasons = []
        if startup_stage in inv['stages_list']:
            score += 35
            reasons.append(f"Invests at {startup_stage} stage")
        if pd.notna(inv['check_size_usd']) and startup_funding_needed_usd:
            ratio = startup_funding_needed_usd / inv['check_size_usd']
            if 0.33 <= ratio <= 3:
                score += 35
                reasons.append(f"Check size aligns (~${inv['check_size_usd']:,.0f})")
        thesis = str(inv['thesis']).lower() if pd.notna(inv['thesis']) else ''
        matched_kw = [kw for kw in keywords if kw in thesis]
        if matched_kw:
            score += 30
            reasons.append(f"Thesis matches: {', '.join(matched_kw)} (specialist fit)")
        results.append({
            'investor_name': inv['name'], 'firm_type': inv['firm_type'],
            'fit_score': score, 'match_reasons': '; '.join(reasons) if reasons else 'No strong match signals',
            'countries': inv['countries']
        })
    return pd.DataFrame(results).sort_values('fit_score', ascending=False)

print("Matching engine loaded.")

def get_sector_success_benchmark(sector, ml_training_df):
    """
    Returns historical success rate for a sector from the ML training data.
    NOTE: This is a reference benchmark from a separate labeled dataset, 
    not a direct prediction for this specific startup - the two datasets 
    don't share a feature space (see project data dictionary).
    """
    sector_data = ml_training_df[ml_training_df['sector'].str.lower() == sector.lower().split(' ')[0]]
    if len(sector_data) == 0:
        return None, 0
    success_rate = sector_data['success'].mean()
    return success_rate, len(sector_data)

df_ml_full = pd.read_csv("data/processed/ml_training_set.csv")
df_ml_full['success'] = df_ml_full['outcome'].isin(['Acquisition', 'IPO']).astype(int)

rate, n = get_sector_success_benchmark('Gaming', df_ml_full)
print(f"Benchmark success rate: {rate:.1%} (n={n})" if rate else "No matching sector benchmark available")

Matching engine loaded.
No matching sector benchmark available


In [4]:
def gather_startup_data(startup_id):
    """Pulls SQL profile, ML readiness score, SHAP factors, and investor matches for one startup."""
    
    # 1. Startup profile + funding rollup from SQL
    profile_query = f"""
    SELECT 
        s.startup_id, s.startup_name, s.sector, s.country, s.founded_year, 
        s.employee_count, s.revenue_status,
        COUNT(f.round_id) as total_rounds,
        ROUND(SUM(f.amount_million_usd), 2) as total_raised_million_usd,
        MAX(f.post_money_valuation_million_usd) as latest_valuation_million_usd,
        MAX(f.funding_stage) as latest_stage
    FROM dim_startup s
    LEFT JOIN fact_funding_rounds f ON s.startup_id = f.startup_id
    WHERE s.startup_id = '{startup_id}'
    GROUP BY s.startup_id;
    """
    profile = pd.read_sql_query(profile_query, conn).iloc[0]
    
    # 2. Sector benchmark - how does this startup compare to sector median?
    benchmark_query = f"""
    SELECT ROUND(AVG(amount_million_usd), 2) as sector_avg_round_million_usd
    FROM fact_funding_rounds
    WHERE sector = '{profile['sector']}';
    """
    sector_avg = pd.read_sql_query(benchmark_query, conn).iloc[0]['sector_avg_round_million_usd']
    
    # 3. Investor matches
    matches = compute_investor_fit(
        profile['sector'], 
        profile['latest_stage'] if pd.notna(profile['latest_stage']) else 'Seed',
        (profile['total_raised_million_usd'] or 1) * 1_000_000,
        df_openvc
    ).head(5)
    
    return {
        'profile': profile,
        'sector_avg_round': sector_avg,
        'top_matches': matches
    }

# Test it
test_data = gather_startup_data('S00001')
print(test_data['profile'])
print(f"\nSector avg round size: ${test_data['sector_avg_round']}M")
print(f"\nTop matches:\n{test_data['top_matches'][['investor_name', 'fit_score']]}")

startup_id                                      S00001
startup_name                                  DataTech
sector                          Gaming & Entertainment
country                                         Mexico
founded_year                                      2020
employee_count                                       9
revenue_status                              Profitable
total_rounds                                         1
total_raised_million_usd                           4.2
latest_valuation_million_usd                     18.18
latest_stage                                      Seed
Name: 0, dtype: object

Sector avg round size: $9.18M

Top matches:
             investor_name  fit_score
29   Nordic Eye Venture...         35
119  Meritech Capital P...         35
76        Pangaea Ventures         35
75                Pontifax         35
30   Monks Hill Venture...         35


In [5]:
print("dim_startup sectors:")
print(sorted(df_startups['sector'].unique()) if 'df_startups' in dir() else pd.read_sql_query("SELECT DISTINCT sector FROM dim_startup", conn)['sector'].tolist())

print("\nml_training_set sectors:")
print(sorted(df_ml_full['sector'].unique()))

dim_startup sectors:
['Gaming & Entertainment', 'Artificial Intelligence', 'SaaS / Enterprise Software', 'Healthcare & Biotech', 'Travel Tech', 'Real Estate Tech', 'Space Tech', 'E-Commerce', 'Web3 / Blockchain', 'Fintech', 'Logistics & Supply Chain', 'Cybersecurity', 'Legal Tech', 'Robotics & Automation', 'Edtech', 'Insurtech', 'Food & AgriTech', 'Climate Tech / Clean Energy', 'HR Tech', 'Social Media / Creator Economy']

ml_training_set sectors:
['AI', 'Climate', 'Crypto', 'Ecommerce', 'Fintech', 'Health', 'SaaS']


In [6]:
SECTOR_ML_MAPPING = {
    'Artificial Intelligence': 'AI',
    'SaaS / Enterprise Software': 'SaaS',
    'Healthcare & Biotech': 'Health',
    'E-Commerce': 'Ecommerce',
    'Web3 / Blockchain': 'Crypto',
    'Fintech': 'Fintech',
    'Climate Tech / Clean Energy': 'Climate'
    # Everything else (Gaming & Entertainment, Travel Tech, Real Estate Tech, 
    # Space Tech, Logistics & Supply Chain, Cybersecurity, Legal Tech, 
    # Robotics & Automation, Edtech, Insurtech, Food & AgriTech, HR Tech, 
    # Social Media / Creator Economy) has no equivalent category in the 
    # ML training set - benchmark unavailable for these, by design.
}

def get_sector_success_benchmark(sector, ml_training_df):
    """
    Returns historical success rate for a sector, if a mapped equivalent 
    exists in the ML training data. Returns None if no mapping exists - 
    this is expected for ~13 of 20 sectors, not an error.
    """
    ml_sector = SECTOR_ML_MAPPING.get(sector)
    if ml_sector is None:
        return None, 0, None
    
    sector_data = ml_training_df[ml_training_df['sector'] == ml_sector]
    success_rate = sector_data['success'].mean()
    return success_rate, len(sector_data), ml_sector

# Test on a mapped sector and an unmapped one
rate, n, ml_sector = get_sector_success_benchmark('Fintech', df_ml_full)
print(f"Fintech → {ml_sector}: {rate:.1%} success rate (n={n})")

rate2, n2, ml_sector2 = get_sector_success_benchmark('Gaming & Entertainment', df_ml_full)
print(f"Gaming & Entertainment → benchmark: {rate2}")

Fintech → Fintech: 43.7% success rate (n=14220)
Gaming & Entertainment → benchmark: None


In [10]:
# ===== CELL 5 (FULL, CORRECTED): PDF GENERATOR =====

from reportlab.lib.pagesizes import letter
from reportlab.lib.units import inch
from reportlab.lib import colors
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from datetime import datetime
import os

def generate_one_pager(startup_id, output_dir="reports"):
    os.makedirs(output_dir, exist_ok=True)
    
    data = gather_startup_data(startup_id)
    profile = data['profile']
    sector_avg = data['sector_avg_round']
    top_matches = data['top_matches']
    
    success_rate, n, ml_sector = get_sector_success_benchmark(profile['sector'], df_ml_full)
    
    filepath = os.path.join(output_dir, f"{startup_id}_{profile['startup_name'].replace(' ', '_')}_due_diligence.pdf")
    doc = SimpleDocTemplate(filepath, pagesize=letter,
                             topMargin=0.6*inch, bottomMargin=0.6*inch,
                             leftMargin=0.6*inch, rightMargin=0.6*inch)
    
    styles = getSampleStyleSheet()
    title_style = ParagraphStyle('TitleStyle', parent=styles['Title'], fontSize=18, spaceAfter=4)
    subtitle_style = ParagraphStyle('SubtitleStyle', parent=styles['Normal'], fontSize=10, textColor=colors.grey, spaceAfter=14)
    section_style = ParagraphStyle('SectionStyle', parent=styles['Heading2'], fontSize=13, spaceBefore=14, spaceAfter=6, textColor=colors.HexColor('#1a3c5e'))
    body_style = styles['Normal']
    note_style = ParagraphStyle('NoteStyle', parent=styles['Normal'], fontSize=8, textColor=colors.grey, spaceAfter=4)
    
    story = []
    
    # --- Header ---
    story.append(Paragraph(f"Due Diligence Snapshot: {profile['startup_name']}", title_style))
    story.append(Paragraph(f"Generated {datetime.now().strftime('%B %d, %Y')} | Startup Investment Intelligence Platform", subtitle_style))
    
    # --- Profile section ---
    story.append(Paragraph("Company Profile", section_style))
    profile_table_data = [
        ['Sector', profile['sector'], 'Country', profile['country']],
        ['Founded', str(profile['founded_year']), 'Employees', str(profile['employee_count'])],
        ['Revenue Status', profile['revenue_status'], 'Latest Stage', profile['latest_stage'] or 'N/A'],
        ['Total Rounds', str(profile['total_rounds']), 'Total Raised', f"${profile['total_raised_million_usd']:.2f}M" if profile['total_raised_million_usd'] else 'N/A'],
        ['Latest Valuation', f"${profile['latest_valuation_million_usd']:.2f}M" if profile['latest_valuation_million_usd'] else 'N/A', '', ''],
    ]
    profile_table = Table(profile_table_data, colWidths=[1.3*inch, 1.9*inch, 1.3*inch, 1.9*inch])
    profile_table.setStyle(TableStyle([
        ('FONTNAME', (0,0), (0,-1), 'Helvetica-Bold'),
        ('FONTNAME', (2,0), (2,-1), 'Helvetica-Bold'),
        ('FONTSIZE', (0,0), (-1,-1), 9),
        ('BOTTOMPADDING', (0,0), (-1,-1), 6),
        ('TOPPADDING', (0,0), (-1,-1), 6),
        ('GRID', (0,0), (-1,-1), 0.5, colors.HexColor('#dddddd')),
        ('BACKGROUND', (0,0), (0,-1), colors.HexColor('#f2f2f2')),
        ('BACKGROUND', (2,0), (2,-1), colors.HexColor('#f2f2f2')),
    ]))
    story.append(profile_table)
    
    # --- Sector benchmark ---
    story.append(Paragraph("Sector Benchmark", section_style))
    story.append(Paragraph(
        f"This startup's average funding round (${profile['total_raised_million_usd']/max(profile['total_rounds'],1):.2f}M) "
        f"compares to a sector average of ${sector_avg:.2f}M across all {profile['sector']} rounds in the platform.", body_style
    ))
    
    if success_rate is not None:
        story.append(Paragraph(
            f"<b>Historical success benchmark:</b> {success_rate:.1%} of startups in a comparable sector "
            f"({ml_sector}, n={n:,}) reached a successful outcome (Acquisition or IPO) in a separate labeled study.", body_style
        ))
    else:
        story.append(Paragraph(
            "<b>Historical success benchmark:</b> Not available - no validated mapping exists between this "
            "sector and the ML training dataset's categories.", body_style
        ))
    story.append(Paragraph(
        "Note: This benchmark reflects sector-level patterns from a separate labeled dataset, not a direct "
        "prediction for this specific startup, since the two data sources do not share a common feature space.",
        note_style
    ))
    
    # --- Investor matches ---
    story.append(Paragraph("Top Investor Matches", section_style))
    match_table_data = [['Investor', 'Type', 'Fit Score', 'Match Basis']]
    for _, row in top_matches.iterrows():
        reasons_short = row['match_reasons'][:60] + ('...' if len(row['match_reasons']) > 60 else '')
        match_table_data.append([row['investor_name'], row['firm_type'], str(row['fit_score']), reasons_short])
    
    match_table = Table(match_table_data, colWidths=[2.0*inch, 0.9*inch, 0.6*inch, 2.9*inch])
    match_table.setStyle(TableStyle([
        ('FONTNAME', (0,0), (-1,0), 'Helvetica-Bold'),
        ('FONTSIZE', (0,0), (-1,-1), 8),
        ('BOTTOMPADDING', (0,0), (-1,-1), 5),
        ('TOPPADDING', (0,0), (-1,-1), 5),
        ('GRID', (0,0), (-1,-1), 0.5, colors.HexColor('#dddddd')),
        ('BACKGROUND', (0,0), (-1,0), colors.HexColor('#1a3c5e')),
        ('TEXTCOLOR', (0,0), (-1,0), colors.white),
    ]))
    story.append(match_table)
    
    # --- Footer / data disclosure ---
    story.append(Spacer(1, 20))
    story.append(Paragraph(
        "Data sources: funding/investor data from a simulated relational dataset; investor thesis data from "
        "OpenVC (real investor names, self-reported theses); sector success benchmarks from a separately "
        "labeled synthetic dataset. This report is a portfolio demonstration and does not constitute investment advice.",
        note_style
    ))
    
    doc.build(story)
    print(f"Report saved: {filepath}")
    return filepath

In [11]:
generate_one_pager('S00001')

Report saved: reports/S00001_DataTech_due_diligence.pdf


'reports/S00001_DataTech_due_diligence.pdf'

In [12]:
test_ids = ['S00001', 'S00002', 'S00063', 'S01445', 'S00301']  # mix from your earlier query outputs

for sid in test_ids:
    try:
        generate_one_pager(sid)
    except Exception as e:
        print(f"FAILED on {sid}: {e}")

Report saved: reports/S00001_DataTech_due_diligence.pdf
Report saved: reports/S00002_VertexPlatform_due_diligence.pdf
Report saved: reports/S00063_ZenoRobotics_due_diligence.pdf
Report saved: reports/S01445_XenoMind_due_diligence.pdf
Report saved: reports/S00301_RapidLabs_due_diligence.pdf
